In [ ]:
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOGO = "desafio_vendas"
T_SILVER = f"{CATALOGO}.silver.vendas"
G = f"{CATALOGO}.gold"
T_QUALIDADE = f"{CATALOGO}.ops.qualidade_resultados"
T_LOG = f"{CATALOGO}.ops.log_pipeline"

ID_LOTE = str(uuid.uuid4())
INICIO = datetime.now()

# Premissas de negócio (documentadas)
STATUS_RECEITA = ["Aprovada"]
STATUS_DEMANDA = ["Aprovada", "Pendente"]
REGIAO = {"PA": "Norte", "CE": "Nordeste", "BA": "Nordeste", "PE": "Nordeste",
          "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
          "PR": "Sul", "RS": "Sul", "SC": "Sul"}

s = spark.table(T_SILVER)

def mapa_para_expr(coluna, mapa, padrao):
    expr = F.lit(padrao)
    for k, v in mapa.items():
        expr = F.when(F.col(coluna) == k, F.lit(v)).otherwise(expr)
    return expr

def com_sk(df, nome_sk, ordem):
    """Chave substituta determinística (ordenada pela chave natural)."""
    return df.select(F.row_number().over(Window.orderBy(*ordem)).alias(nome_sk), *df.columns)

def escrever(df, nome, comentario):
    destino = f"{G}.{nome}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(destino)
    spark.sql(f"COMMENT ON TABLE {destino} IS '{comentario}'")
    print(f"{destino}: {spark.table(destino).count()} linhas")

In [ ]:
print(s.agg(F.count("*").alias("vendas"), F.countDistinct("cliente").alias("clientes")).toPandas())

conf = s.groupBy("cliente").agg(F.count("*").alias("vendas"),
                                F.countDistinct("cliente_sexo").alias("sexos"),
                                F.countDistinct("cliente_cpf_hash").alias("cpfs"))
display(conf.groupBy("vendas").count().orderBy("vendas").limit(10))
print("Clientes com mais de 1 sexo:", conf.filter("sexos > 1").count(),
      "| com mais de 1 CPF:", conf.filter("cpfs > 1").count())

In [ ]:
MESES = ["Janeiro","Fevereiro","Março","Abril","Maio","Junho","Julho","Agosto","Setembro","Outubro","Novembro","Dezembro"]
DIAS = ["Domingo","Segunda-feira","Terça-feira","Quarta-feira","Quinta-feira","Sexta-feira","Sábado"]

# dim_tempo: cobre as datas de venda e de retirada (inclusive as inconsistentes)
datas = spark.sql("SELECT explode(sequence(to_date('2020-12-01'), to_date('2024-03-31'), interval 1 day)) AS data")
d_tempo = datas.select(
    F.date_format("data", "yyyyMMdd").cast("int").alias("sk_tempo"),
    "data",
    F.year("data").alias("ano"),
    F.quarter("data").alias("trimestre"),
    F.month("data").alias("mes"),
    F.element_at(F.array(*[F.lit(m) for m in MESES]), F.month("data")).alias("nome_mes"),
    F.date_format("data", "yyyy-MM").alias("ano_mes"),
    F.trunc("data", "MM").alias("primeiro_dia_mes"),
    F.dayofmonth("data").alias("dia"),
    F.dayofweek("data").alias("dia_semana"),
    F.element_at(F.array(*[F.lit(d) for d in DIAS]), F.dayofweek("data")).alias("nome_dia_semana"),
    F.dayofweek("data").isin(1, 7).alias("flag_fim_de_semana"),
)

d_prod = com_sk(s.select("produto", "tipo_produto").distinct(), "sk_produto", ["produto"])

loc = (s.select(F.coalesce("cidade", F.lit("Não informada")).alias("cidade"),
                F.coalesce("uf", F.lit("NI")).alias("uf")).distinct()
        .withColumn("regiao", mapa_para_expr("uf", REGIAO, "Não informada")))
d_loc = com_sk(loc, "sk_localidade", ["uf", "cidade"])

d_pag = com_sk(s.select("tipo_pagamento").distinct(), "sk_pagamento", ["tipo_pagamento"])
d_status = com_sk(s.select("status_venda", "status_envio").distinct(), "sk_status", ["status_venda", "status_envio"])

# dim_cliente: sexo = valor mais frequente do cliente (em empate, o mais recente); sinaliza conflito
cli_sexo = (s.groupBy("cliente", "cliente_sexo").agg(F.count("*").alias("n"), F.max("data_venda").alias("ultima"))
              .withColumn("rk", F.row_number().over(
                  Window.partitionBy("cliente").orderBy(F.desc("n"), F.desc("ultima"), "cliente_sexo")))
              .filter("rk = 1").select("cliente", "cliente_sexo"))
cli_stats = s.groupBy("cliente").agg(F.countDistinct("cliente_sexo").alias("qtd_sexos_distintos"),
                                     F.count("*").alias("qtd_vendas_historico"))
d_cli = com_sk(cli_sexo.join(cli_stats, "cliente"), "sk_cliente", ["cliente"])

escrever(d_tempo, "dim_tempo", "Dimensao calendario (dia). Cobre datas de venda e de retirada")
escrever(d_prod, "dim_produto", "Dimensao produto e tipo de produto")
escrever(d_loc, "dim_localidade", "Dimensao localidade: cidade, UF e regiao. Nao informada quando ausente")
escrever(d_pag, "dim_pagamento", "Dimensao tipo de pagamento")
escrever(d_status, "dim_status", "Dimensao junk: combinacao de status da venda e status do envio")
escrever(d_cli, "dim_cliente", "Dimensao cliente (sem PII). Sexo = valor mais frequente do cliente")

# Recarrega as dimensões gravadas (garante que a fato usa as mesmas chaves)
d_prod, d_loc, d_pag = (spark.table(f"{G}.{t}") for t in ["dim_produto", "dim_localidade", "dim_pagamento"])
d_status, d_cli = spark.table(f"{G}.dim_status"), spark.table(f"{G}.dim_cliente")

In [ ]:
base = (s.withColumn("cidade", F.coalesce("cidade", F.lit("Não informada")))
         .withColumn("uf", F.coalesce("uf", F.lit("NI"))))

j = (base
     .join(d_prod, ["produto", "tipo_produto"], "left")
     .join(d_loc.select("sk_localidade", "cidade", "uf"), ["cidade", "uf"], "left")
     .join(d_pag, "tipo_pagamento", "left")
     .join(d_status, ["status_venda", "status_envio"], "left")
     .join(d_cli.select("sk_cliente", "cliente"), "cliente", "left"))

vb = F.col("qtd_vendida") * F.col("valor_unitario")
fato = j.select(
    "id_venda",
    F.date_format("data_venda", "yyyyMMdd").cast("int").alias("sk_tempo_venda"),
    F.date_format("data_retirada", "yyyyMMdd").cast("int").alias("sk_tempo_retirada"),
    "sk_produto", "sk_cliente", "sk_localidade", "sk_pagamento", "sk_status",
    "qtd_vendida", "valor_unitario",
    F.round(vb, 2).cast("decimal(14,2)").alias("valor_bruto"),
    "desconto_pct",
    F.round(vb * F.col("desconto_pct") / 100, 2).cast("decimal(14,2)").alias("valor_desconto"),
    F.round(vb * (1 - F.col("desconto_pct") / 100), 2).cast("decimal(14,2)").alias("valor_liquido"),
    "valor_frete", "avaliacao_produto", "desconto_aplicado", "item_promocional",
    F.when(F.col("data_retirada") >= F.col("data_venda"),
           F.datediff("data_retirada", "data_venda")).alias("dias_ate_retirada"),
    (F.col("data_retirada") < F.col("data_venda")).alias("flag_retirada_inconsistente"),
    F.col("status_venda").isin(*STATUS_RECEITA).alias("flag_receita"),
    F.col("status_venda").isin(*STATUS_DEMANDA).alias("flag_demanda"),
    F.size("alertas").alias("qtd_alertas"),
    "_id_lote_silver",
    F.lit(ID_LOTE).alias("_id_lote_gold"),
    F.current_timestamp().alias("_data_processamento"),
)

escrever(fato, "fato_vendas", "Fato de vendas, grao uma venda. Medidas: quantidade, valor bruto, desconto, valor liquido e frete")

In [ ]:
f  = spark.table(f"{G}.fato_vendas")
t  = spark.table(f"{G}.dim_tempo").select(F.col("sk_tempo").alias("sk_tempo_venda"), "ano_mes", "primeiro_dia_mes")
p  = spark.table(f"{G}.dim_produto")
st = spark.table(f"{G}.dim_status").select("sk_status", "status_venda")

def soma_se(cond, valor):
    return F.sum(F.when(cond, valor).otherwise(0))

agg = (f.join(t, "sk_tempo_venda").join(p, "sk_produto").join(st, "sk_status")
        .groupBy("primeiro_dia_mes", "ano_mes", "produto", "tipo_produto")
        .agg(
            F.count("*").alias("qtd_vendas"),
            soma_se(F.col("status_venda") == "Aprovada", F.lit(1)).alias("qtd_vendas_aprovadas"),
            soma_se(F.col("status_venda") == "Pendente", F.lit(1)).alias("qtd_vendas_pendentes"),
            soma_se(F.col("status_venda") == "Cancelada", F.lit(1)).alias("qtd_vendas_canceladas"),
            soma_se(F.col("status_venda") == "Desconhecido", F.lit(1)).alias("qtd_vendas_desconhecidas"),
            soma_se(F.col("flag_demanda"), F.col("qtd_vendida")).alias("unidades_demanda"),
            soma_se(F.col("flag_receita"), F.col("qtd_vendida")).alias("unidades_aprovadas"),
            soma_se(F.col("flag_receita"), F.col("valor_bruto")).alias("receita_bruta"),
            soma_se(F.col("flag_receita"), F.col("valor_desconto")).alias("desconto_total"),
            soma_se(F.col("flag_receita"), F.col("valor_liquido")).alias("receita_liquida"),
            soma_se(F.col("flag_receita"), F.col("valor_frete")).alias("frete_total"),
            F.round(F.avg(F.when(F.col("status_venda") != "Cancelada", F.col("avaliacao_produto"))), 2).alias("avaliacao_media"),
        )
        .withColumn("ticket_medio", F.round(F.col("receita_liquida") / F.col("qtd_vendas_aprovadas"), 2))
        .withColumn("pct_cancelamento", F.round(100 * F.col("qtd_vendas_canceladas") / F.col("qtd_vendas"), 2)))

escrever(agg, "agg_vendas_mensal_produto",
         "Agregacao mensal por produto. Receita = aprovadas. Demanda = aprovadas + pendentes. Base do BI e da previsao")
display(spark.table(f"{G}.agg_vendas_mensal_produto").orderBy("produto", "primeiro_dia_mes").limit(12))

In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {G}.vw_vendas_analitica AS
SELECT f.id_venda,
       tv.data AS data_venda, tv.ano, tv.trimestre, tv.mes, tv.nome_mes, tv.ano_mes,
       tr.data AS data_retirada,
       p.produto, p.tipo_produto,
       c.cliente, c.cliente_sexo,
       l.cidade, l.uf, l.regiao,
       pg.tipo_pagamento,
       s.status_venda, s.status_envio,
       f.qtd_vendida, f.valor_unitario, f.valor_bruto, f.desconto_pct, f.valor_desconto,
       f.valor_liquido, f.valor_frete, f.avaliacao_produto,
       f.desconto_aplicado, f.item_promocional, f.dias_ate_retirada,
       f.flag_retirada_inconsistente, f.flag_receita, f.flag_demanda, f.qtd_alertas
FROM {G}.fato_vendas f
JOIN {G}.dim_tempo tv ON f.sk_tempo_venda = tv.sk_tempo
LEFT JOIN {G}.dim_tempo tr ON f.sk_tempo_retirada = tr.sk_tempo
JOIN {G}.dim_produto p ON f.sk_produto = p.sk_produto
JOIN {G}.dim_cliente c ON f.sk_cliente = c.sk_cliente
JOIN {G}.dim_localidade l ON f.sk_localidade = l.sk_localidade
JOIN {G}.dim_pagamento pg ON f.sk_pagamento = pg.sk_pagamento
JOIN {G}.dim_status s ON f.sk_status = s.sk_status
""")
spark.sql(f"COMMENT ON TABLE {G}.vw_vendas_analitica IS 'View analitica desnormalizada (fato + dimensoes) para BI e ciencia de dados. Sem PII'")

In [ ]:
fato = spark.table(f"{G}.fato_vendas")
n_silver, n_fato = s.count(), fato.count()

sks = ["sk_tempo_venda", "sk_produto", "sk_cliente", "sk_localidade", "sk_pagamento", "sk_status"]
nulos_sk = fato.filter(" OR ".join(f"{c} IS NULL" for c in sks)).count()
orfaos_tempo = fato.join(spark.table(f"{G}.dim_tempo"), fato.sk_tempo_venda == F.col("sk_tempo"), "left_anti").count()
dup_id = n_fato - fato.select("id_venda").distinct().count()
dif_qtd = abs(fato.agg(F.sum("qtd_vendida")).collect()[0][0] - s.agg(F.sum("qtd_vendida")).collect()[0][0])
negativos = fato.filter("valor_liquido < 0 OR valor_bruto < 0").count()

verificacoes = [
    ("G01", "contagem da fato diferente da Silver", abs(n_fato - n_silver)),
    ("G02", "id_venda duplicado na fato", dup_id),
    ("G03", "chaves substitutas nulas na fato", nulos_sk),
    ("G04", "data de venda sem correspondente em dim_tempo", orfaos_tempo),
    ("G05", "soma de qtd_vendida diverge da Silver", int(dif_qtd)),
    ("G06", "valores monetarios negativos", negativos),
]
linhas = [(ID_LOTE, "gold", rid, desc, "ERRO", int(q), int(n_fato),
           round(100.0 * q / n_fato, 2), datetime.now()) for rid, desc, q in verificacoes]
(spark.createDataFrame(linhas,
    "id_lote string, etapa string, regra_id string, descricao string, severidade string, "
    "qtd_afetada long, qtd_total long, pct_afetado double, data_execucao timestamp")
 .write.mode("append").saveAsTable(T_QUALIDADE))

ok = all(q == 0 for _, _, q in verificacoes)
(spark.createDataFrame(
    [(ID_LOTE, "gold", f"{G}.fato_vendas", n_silver, n_fato, "SUCESSO" if ok else "DIVERGENCIA",
      INICIO, datetime.now(), 0, 0)],
    "id_lote string, etapa string, tabela_destino string, qtd_lida long, qtd_gravada long, status string, "
    "inicio timestamp, fim timestamp, qtd_rejeitada long, qtd_duplicada long")
 .write.mode("append").saveAsTable(T_LOG))

print(f"Silver={n_silver} | Fato={n_fato} | Todas as verificações OK = {ok}")
display(spark.table(T_QUALIDADE).filter(F.col("id_lote") == ID_LOTE).orderBy("regra_id"))
display(spark.table(f"{G}.vw_vendas_analitica").limit(10))

In [ ]:
import json
dbutils.notebook.exit(json.dumps({
    "id_lote": ID_LOTE, "qtd_lida": n_silver, "qtd_gravada": n_fato,
    "qtd_rejeitada": 0, "qtd_duplicada": 0,
    "status": "SUCESSO" if ok else "DIVERGENCIA"}))